# 03 — Embedding Model Fine-tuning

**Goal**: Learn how to fine-tune embedding models with contrastive learning, evaluate retrieval quality, and integrate into a RAG pipeline.

**Prerequisites**: Familiarity with transformers, PEFT (LoRA), and vector similarity concepts.

In [ ]:
!pip install sentence-transformers datasets faiss-cpu peft matplotlib scikit-learn torch -q

## 1. Embedding Models Overview

Modern embedding models map text to dense vectors where semantic similarity equals cosine distance.

| Model | Size | Dim | Max Tokens | Notes |
|-------|------|-----|------------|-------|
| **BGE-small-en-v1.5** | 33M | 384 | 512 | Lightweight, good baseline |
| **BGE-large-en-v1.5** | 335M | 1024 | 512 | Strong general-purpose, MTEB top-ranked |
| **E5-mistral-7b-instruct** | 7B | 4096 | 32768 | Task-specific instructions, very strong |
| **Jina-embeddings-v3** | 570M | 1024 | 8192 | Multilingual, long-context, task-aware |
| **multilingual-e5-large** | 560M | 1024 | 512 | Multilingual (100+ languages) |

**Our choice**: `BAAI/bge-small-en-v1.5` — small enough for CPU fine-tuning, strong enough to show improvements.

## 2. Load the Base Embedding Model

In [ ]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.datasets import NoDuplicatesDataLoader

BASE_MODEL = "BAAI/bge-small-en-v1.5"

print(f"Loading {BASE_MODEL}...")
model = SentenceTransformer(BASE_MODEL)
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {model.max_seq_length}")

# Quick sanity check
test_emb = model.encode("How do I reset my password?")
print(f"Test embedding shape: {test_emb.shape}")
print(f"Norm: {np.linalg.norm(test_emb):.3f}")

## 3. Why Fine-Tune Embeddings?

General embedding models are trained on diverse web data. In specialized domains (medical, legal, customer support), they miss domain-specific semantic relationships.

**Example**:
- General model: "heart attack" vs "myocardial infarction" — moderately similar
- Medical fine-tuned: very high similarity — better retrieval for medical RAG

**Benefits**: Better retrieval for RAG, domain terminology understanding, cost savings from fewer tokens in prompts.

## 4. Prepare Contrastive Learning Data: Positive Pairs

For contrastive fine-tuning, we need (query, positive_document) pairs. These come from click logs, Q&A pairs, or synthetic generation.

In [ ]:
# Domain: Tech Support Knowledge Base — (query, relevant_document) pairs
tech_support_data = [
    ("My WiFi keeps disconnecting every few minutes",
     "WiFi disconnection is often caused by router interference, outdated drivers, or signal congestion. Restart your router and update network drivers."),
    ("How do I fix slow internet on my laptop?",
     "Fix slow internet by: 1) Checking background downloads, 2) Moving closer to the router, 3) Switching to 5GHz band, 4) Running a speed test."),
    ("I forgot my password and can't log in",
     "Click Forgot Password on the login screen, check your recovery email for a reset link, and create a new password with at least 8 characters including a number and symbol."),
    ("Account locked after too many login attempts",
     "Accounts are temporarily locked after 5 failed login attempts. Wait 15 minutes before trying again, or use the Forgot Password option to reset immediately."),
    ("Not receiving emails in my inbox",
     "Check your spam folder first. Verify email filters and blocked senders. If using Gmail, check the All Mail view. Also verify mailbox storage is not full."),
    ("Why are my emails going to spam folders?",
     "Emails land in spam due to: missing SPF/DKIM records, spammy subject/body content, new domain without reputation, or high complaint rates."),
    ("Printer shows offline but it's plugged in",
     "If printer shows offline: 1) Restart printer and computer, 2) Check printer is set as default, 3) Disable Use Printer Offline in properties, 4) Reinstall drivers."),
    ("How do I clear a paper jam?",
     "Turn off the printer, open the access door, gently pull paper in the direction of the paper path, check rollers for torn pieces, close door, restart."),
    ("Error installing software: permission denied",
     "Windows: Right-click installer > Run as Administrator. Mac: System Preferences > Security & Privacy > Allow apps from identified developers."),
    ("Software crashes on startup after installation",
     "Startup crashes may be from missing dependencies, corrupted installation files, or antivirus conflicts. Reinstall with antivirus temporarily disabled."),
    ("How do I set up email forwarding?",
     "Settings > See all settings > Forwarding and POP/IMAP > Add a forwarding address > Verify with confirmation email > Enable forwarding."),
    ("Can't connect to WiFi after system update",
     "After system updates, go to Device Manager > Network Adapters > Right-click adapter > Update driver > Browse my computer > Let me pick."),
    ("How to uninstall a program completely?",
     "Windows: Settings > Apps > Installed apps > Uninstall. For complete removal, use Revo Uninstaller to remove leftover registry entries."),
    ("Printer prints blank pages",
     "Blank pages can be caused by: empty ink/toner, clogged print heads, incorrect paper type setting, or software issue. Run the built-in cleaning cycle."),
]

print(f"Created {len(tech_support_data)} (query, document) pairs")
for i in [0, 3, 6]:
    q, doc = tech_support_data[i]
    print(f"\nQuery: '{q}'")
    print(f"Doc:   '{doc[:80]}...'")

## 5. Hard Negative Mining Strategy

**What are hard negatives?** Documents that are similar to the query but NOT relevant — they teach fine-grained distinctions.

**Example**: For "How to reset password", a hard negative is "How to reset network settings" — topically similar but wrong answer.

**Mining strategies**:
- In-batch negatives: other queries' positives in the same batch (free, used by MultipleNegativesRankingLoss)
- Top-k retrieval: use current model to retrieve near-miss documents
- LLM-generated: ask GPT to generate plausible-but-wrong answers

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

class HardNegativeMiner:
    """Mine hard negatives using the current embedding model."""

    def __init__(self, model):
        self.model = model
        self.doc_texts = []
        self.doc_embeddings = None

    def index_documents(self, documents):
        self.doc_texts = documents
        self.doc_embeddings = self.model.encode(documents, show_progress_bar=True)
        print(f"Indexed {len(documents)} documents.")

    def find_hard_negatives(self, query, positive_idx, top_k=5):
        query_emb = self.model.encode([query])[0]
        similarities = cosine_similarity([query_emb], self.doc_embeddings)[0]
        ranked = np.argsort(similarities)[::-1]
        hard_negs = []
        for idx in ranked:
            if idx != positive_idx and len(hard_negs) < top_k:
                hard_negs.append((int(idx), float(similarities[idx]), self.doc_texts[idx][:100]))
        return hard_negs

# Extract unique documents
unique_docs = list(set(doc for _, doc in tech_support_data))
print(f"Unique documents: {len(unique_docs)}")

# Mine hard negatives
miner = HardNegativeMiner(model)
miner.index_documents(unique_docs)

print("\n=== Hard Negative Examples ===")
q, pos_doc = tech_support_data[0]
doc_idx = unique_docs.index(pos_doc)
hn = miner.find_hard_negatives(q, doc_idx, top_k=3)
print(f"Query: '{q}'")
print(f"  Positive: '{pos_doc[:80]}...'")
print(f"  Hard negatives:")
for idx, sim, text in hn:
    print(f"    (sim={sim:.3f}) '{text}...'")

## 6. LoRA for Embedding Model with PEFT

LoRA adds small trainable matrices to attention layers — ~1% of full fine-tuning parameters. Memory-efficient and fast.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "dense"],
    bias="none",
)

# Get the underlying transformer
underlying = model._first_module()
if hasattr(underlying, 'auto_model'):
    underlying = underlying.auto_model

peft_model = get_peft_model(underlying, lora_config)
peft_model.print_trainable_parameters()

# Replace transformer in SentenceTransformer with PEFT version
for i, module in enumerate(model.modules()):
    if hasattr(module, 'auto_model'):
        module.auto_model = peft_model
        print(f"Replaced transformer at index {i} with PEFT model.")
        break

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 7. MultipleNegativesRankingLoss Training

**How it works**: For N (query, positive_doc) pairs in a batch, each query is compared against ALL documents. The loss encourages the correct pair to have higher similarity than all other combinations. Other queries' positives serve as free in-batch negatives.

In [ ]:
# Prepare InputExamples
train_examples = [InputExample(texts=[q, doc]) for q, doc in tech_support_data]
print(f"Training examples: {len(train_examples)}")

# Create dataloader (NoDuplicatesDataLoader removes duplicate entries)
train_dataloader = NoDuplicatesDataLoader(train_examples, batch_size=4)

# Set up MultipleNegativesRankingLoss
train_loss = losses.MultipleNegativesRankingLoss(model)

# Train
print("Starting fine-tuning (5 epochs, batch_size=4)...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=5,
    warmup_steps=10,
    optimizer_params={'lr': 2e-4},
    output_path='./bge-small-tech-support-lora',
    save_best_model=True,
    show_progress_bar=True,
)
print("Fine-tuning complete!")

## 8. Retrieval Evaluation: Hit@k and MRR

The most important RAG metric: does the right document appear in the top-K retrieved results?

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

class RetrievalEvaluator:
    """Evaluate retrieval performance with hit rate and MRR."""

    def __init__(self, model, documents):
        self.model = model
        self.documents = documents
        self.doc_embeddings = model.encode(documents, show_progress_bar=True, convert_to_tensor=True)

    def evaluate(self, queries, true_doc_indices, k_values=[1, 3, 5]):
        query_embeddings = self.model.encode(queries, show_progress_bar=False, convert_to_tensor=True)
        similarities = cosine_similarity(query_embeddings.cpu(), self.doc_embeddings.cpu())
        results = {f'hit@{k}': [] for k in k_values}
        results['mrr'] = []
        for i, (sims, true_idx) in enumerate(zip(similarities, true_doc_indices)):
            ranked = np.argsort(sims)[::-1]
            rank = int(np.where(ranked == true_idx)[0][0]) + 1
            for k in k_values:
                results[f'hit@{k}'].append(1.0 if rank <= k else 0.0)
            results['mrr'].append(1.0 / rank)
        return {key: float(np.mean(values)) for key, values in results.items()}

# Define test queries (paraphrased, not seen during training)
test_queries = [
    "WiFi drops connection repeatedly",
    "internet connection is very sluggish",
    "locked out of my account",
    "emails not showing up in Gmail",
    "printer appears disconnected",
]

# True document indices in unique_docs for each query
true_indices = [
    unique_docs.index(tech_support_data[0][1]),
    unique_docs.index(tech_support_data[1][1]),
    unique_docs.index(tech_support_data[3][1]),
    unique_docs.index(tech_support_data[4][1]),
    unique_docs.index(tech_support_data[6][1]),
]

# Evaluate before fine-tuning (base model)
base_evaluator = RetrievalEvaluator(model, unique_docs)
base_results = base_evaluator.evaluate(test_queries, true_indices)

# Evaluate after fine-tuning
ft_model = SentenceTransformer('./bge-small-tech-support-lora')
ft_evaluator = RetrievalEvaluator(ft_model, unique_docs)
ft_results = ft_evaluator.evaluate(test_queries, true_indices)

# Comparison
print(f"{'Metric':<12} {'Base':<12} {'Fine-tuned':<12} {'Change':<12}")
print("-" * 48)
for key in sorted(base_results.keys()):
    b = base_results[key]
    ft = ft_results[key]
    print(f"{key:<12} {b:<12.4f} {ft:<12.4f} {ft-b:+<12.4f}")

## 9. t-SNE Visualization of Embeddings

See how fine-tuning changes the embedding space — do related documents cluster better?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

sns.set_style("whitegrid")

# Assign categories
category_labels = []
for doc in unique_docs:
    if 'WiFi' in doc or 'internet' in doc or 'network' in doc:
        category_labels.append('WiFi/Network')
    elif 'password' in doc or 'login' in doc or 'Account' in doc:
        category_labels.append('Account')
    elif 'email' in doc or 'inbox' in doc or 'spam' in doc:
        category_labels.append('Email')
    elif 'printer' in doc or 'paper' in doc:
        category_labels.append('Printer')
    elif 'install' in doc or 'uninstall' in doc or 'crashes' in doc:
        category_labels.append('Software')
    else:
        category_labels.append('Other')

# Encode with both models
before_emb = model.encode(unique_docs)
after_emb = ft_model.encode(unique_docs)

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
combined = np.vstack([before_emb, after_emb])
combined_2d = tsne.fit_transform(combined)
before_2d = combined_2d[:len(unique_docs)]
after_2d = combined_2d[len(unique_docs):]

# Plot
unique_labels = list(set(category_labels))
colors = sns.color_palette("husl", len(unique_labels))
label_to_color = dict(zip(unique_labels, colors))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
for lbl in unique_labels:
    mask = [l == lbl for l in category_labels]
    ax1.scatter(before_2d[mask, 0], before_2d[mask, 1], c=[label_to_color[lbl]], label=lbl, alpha=0.7, s=80)
    ax2.scatter(after_2d[mask, 0], after_2d[mask, 1], c=[label_to_color[lbl]], label=lbl, alpha=0.7, s=80)
ax1.set_title('Before Fine-Tuning (Base BGE-small)', fontsize=12)
ax2.set_title('After Fine-Tuning (Domain-Adapted)', fontsize=12)
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
fig.suptitle('t-SNE: Embedding Clustering Before vs After', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 10. Cosine Similarity Analysis: Margin Improvement

Compare similarity scores for relevant vs. irrelevant pairs before and after fine-tuning.

In [ ]:
# Create (query, positive, negative) triplets
analysis_queries = [
    ("WiFi keeps disconnecting", tech_support_data[0][1], tech_support_data[6][1]),  # WiFi query, WiFi doc, Printer doc
    ("forgot my password", tech_support_data[2][1], tech_support_data[6][1]),       # password query, password doc, printer doc
    ("emails going to spam", tech_support_data[5][1], tech_support_data[0][1]),      # email query, email doc, wifi doc
    ("software won't install", tech_support_data[8][1], tech_support_data[2][1]),    # software query, software doc, password doc
]

results = {'base_pos': [], 'base_neg': [], 'ft_pos': [], 'ft_neg': []}
for q, pos, neg in analysis_queries:
    q_emb_base = model.encode([q])
    results['base_pos'].append(cosine_similarity(q_emb_base, model.encode([pos]))[0][0])
    results['base_neg'].append(cosine_similarity(q_emb_base, model.encode([neg]))[0][0])
    q_emb_ft = ft_model.encode([q])
    results['ft_pos'].append(cosine_similarity(q_emb_ft, ft_model.encode([pos]))[0][0])
    results['ft_neg'].append(cosine_similarity(q_emb_ft, ft_model.encode([neg]))[0][0])

# Box plot
fig, ax = plt.subplots(figsize=(10, 5))
x_labels = ['Base Positive', 'Base Negative', 'FT Positive', 'FT Negative']
data = [results['base_pos'], results['base_neg'], results['ft_pos'], results['ft_neg']]
colors_box = ['#2ecc71', '#e74c3c', '#27ae60', '#c0392b']
bp = ax.boxplot(data, labels=x_labels, patch_artist=True)
for patch, c in zip(bp['boxes'], colors_box):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title('Similarity Scores: Base vs Fine-Tuned', fontsize=14)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

margin_base = np.mean(results['base_pos']) - np.mean(results['base_neg'])
margin_ft = np.mean(results['ft_pos']) - np.mean(results['ft_neg'])
print(f"Separation margin (base):      {margin_base:.4f}")
print(f"Separation margin (fine-tuned): {margin_ft:.4f}")
print(f"Improvement:                    {margin_ft - margin_base:+.4f}")

## 11. Integration with RAG Pipeline (LangChain)

Plug fine-tuned embeddings into a real retrieval pipeline.

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document as LangchainDoc

# Create LangChain-compatible embeddings
ft_embeddings = HuggingFaceEmbeddings(
    model_name='./bge-small-tech-support-lora',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

# Create LangChain documents
lc_docs = [
    LangchainDoc(page_content=doc, metadata={"source": f"kb_{i}"})
    for i, doc in enumerate(unique_docs)
]

# Build FAISS vector store
vector_store = FAISS.from_documents(lc_docs, ft_embeddings)
print(f"Vector store created with {len(lc_docs)} documents.\n")

# Test RAG retrieval
rag_queries = [
    "My WiFi keeps disconnecting, what should I do?",
    "How do I reset my password?",
    "Printer is offline but it's connected",
]
for query in rag_queries:
    print(f"Query: '{query}'")
    docs = vector_store.similarity_search_with_score(query, k=2)
    for rank, (doc, score) in enumerate(docs, 1):
        print(f"  #{rank} (dist={score:.4f}): {doc.page_content[:80]}...")
    print()

## 12. Export the Fine-Tuned Model

Merge LoRA weights into the base model for production deployment.

In [ ]:
import os

# Option 1: Save as SentenceTransformer format (with LoRA adapter)
ft_model.save('./bge-small-tech-support-export')
print("Exported fine-tuned model (with LoRA).")

# Option 2: Merge LoRA into base weights (no adapter overhead at inference)
try:
    merged_transformer = peft_model.merge_and_unload()
    for module in ft_model.modules():
        if hasattr(module, 'auto_model'):
            module.auto_model = merged_transformer
            break
    ft_model.save('./bge-small-tech-support-merged')
    print("Exported merged model (LoRA weights absorbed).")
except Exception as e:
    print(f"Merge note: {e}. Model is still usable with LoRA.")

# Report sizes
for path in ['./bge-small-tech-support-lora', './bge-small-tech-support-export']:
    if os.path.exists(path):
        total = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fns in os.walk(path) for f in fns)
        print(f"  {path}: {total / 1024**2:.1f} MB")

## 13. Exercise: Fine-Tune on Your Own Domain

Create a small domain dataset (8+ pairs), fine-tune BGE-small, and measure the retrieval improvement.

In [ ]:
# ===== YOUR TURN: Fine-tune on your own domain =====

# Step 1: Create your own domain dataset
my_domain_data = [
    ("How do I make a perfect omelette?",
     "Whisk 2-3 eggs with salt, heat butter in a non-stick pan over medium heat, pour eggs, cook until edges set, add fillings, fold and serve."),
    ("What temperature for chocolate chip cookies?",
     "Bake at 350F (175C) for 10-12 minutes until edges are golden. Cool on baking sheet 5 minutes before transferring."),
    ("How long to boil pasta al dente?",
     "Thin pasta 8-10 min, thick pasta 10-12 min, fresh pasta 2-3 min. Always taste-test a minute before the minimum time."),
    ("What can substitute for eggs in baking?",
     "1/4 cup applesauce, 1 mashed banana, 1 tbsp flaxseed + 3 tbsp water, or 1/4 cup yogurt."),
    ("How to tell if chicken is fully cooked?",
     "Internal temperature of 165F at thickest part. Juices should run clear. Use a meat thermometer."),
    ("Why did my bread not rise?",
     "Dead yeast, water too hot, water too cold, not enough kneading, or rising in a cold spot. Proof yeast in warm water (105-110F)."),
    ("How to sharpen a chef's knife at home?",
     "Use a whetstone (1000/6000 grit), soak 10-15 min, hold at 15-20 degree angle, push blade from heel to tip, repeat 10-15 times per side."),
    ("What's the difference between baking soda and baking powder?",
     "Baking soda needs acid to activate. Baking powder contains baking soda + cream of tartar, activates with liquid and heat."),
]
print(f"Custom dataset: {len(my_domain_data)} pairs")

# Step 2: Load fresh base model and apply LoRA
my_model = SentenceTransformer(BASE_MODEL)
my_underlying = my_model._first_module()
if hasattr(my_underlying, 'auto_model'):
    my_underlying = my_underlying.auto_model
my_peft = get_peft_model(my_underlying, lora_config)
for module in my_model.modules():
    if hasattr(module, 'auto_model'):
        module.auto_model = my_peft
        break

# Step 3: Train
my_examples = [InputExample(texts=[q, doc]) for q, doc in my_domain_data]
my_dataloader = NoDuplicatesDataLoader(my_examples, batch_size=4)
my_loss = losses.MultipleNegativesRankingLoss(my_model)
print("Training on your domain...")
my_model.fit(
    train_objectives=[(my_dataloader, my_loss)],
    epochs=10,
    warmup_steps=5,
    optimizer_params={'lr': 2e-4},
    output_path='./bge-small-my-domain',
    show_progress_bar=True,
)

# Step 4: Evaluate
my_docs = [doc for _, doc in my_domain_data]
test_q = [
    "How do I cook an omelette properly?",
    "What oven temp for cookies?",
    "How many minutes for spaghetti?",
    "Can I replace eggs in a cake recipe?",
]
true_idx = [0, 1, 2, 3]
base_eval = RetrievalEvaluator(SentenceTransformer(BASE_MODEL), my_docs)
base_r = base_eval.evaluate(test_q, true_idx)
ft_eval = RetrievalEvaluator(SentenceTransformer('./bge-small-my-domain'), my_docs)
ft_r = ft_eval.evaluate(test_q, true_idx)

print(f"\n{'Metric':<12} {'Base':<12} {'Fine-tuned':<12} {'Change':<12}")
print("-" * 48)
for key in sorted(base_r.keys()):
    print(f"{key:<12} {base_r[key]:<12.4f} {ft_r[key]:<12.4f} {ft_r[key]-base_r[key]:+<12.4f}")
print(f"\nHit@1 improvement: {ft_r['hit@1'] - base_r['hit@1']:+.1%}")

## Summary

| Stage | What You Did |
|-------|--------------|
| **Data Prep** | Created positive (query, doc) pairs and mined hard negatives |
| **Model Setup** | Loaded BGE-small, applied LoRA for efficient fine-tuning |
| **Training** | Used MultipleNegativesRankingLoss for contrastive learning |
| **Evaluation** | Measured hit@k and MRR before vs after |
| **Visualization** | t-SNE plots showing embedding clustering improvement |
| **Deployment** | Exported model, integrated with LangChain FAISS RAG pipeline |

**Key takeaways**:
1. LoRA makes fine-tuning practical: ~1% of params trainable, runs on consumer GPUs
2. In-batch negatives from MultipleNegativesRankingLoss work well without explicit hard negative mining
3. Even 10-15 domain-specific pairs can measurably improve retrieval
4. Always measure before/after hit rate — it's your ground truth for RAG quality
5. SentenceTransformer format integrates seamlessly with LangChain and LlamaIndex